In [ ]:
"""
台股PSO-BIGRU預測系統
包含：PSO參數優化
不包含：HFSLS特徵選擇（使用全部特徵）
用於消融實驗：驗證PSO優化的效果
"""

import math
import os
import pandas as pd
import numpy as np
from collections import deque
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import warnings
import time
from datetime import timedelta
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ============================================================
# 全局配置
# ============================================================
scaler = MinMaxScaler()
window = 20
loss = "mse"
measure = [[]]
true_pre = [[], []]
best_result_buffer = None

# ============================================================
# 進度追蹤器
# ============================================================
class ProgressTracker:
    def __init__(self, total_stocks):
        self.total_stocks = total_stocks
        self.start_time = time.time()
        self.stock_start_time = None
        
    def start_stock(self, stock_name, stock_idx):
        self.stock_start_time = time.time()
        print(f"\n{'='*70}")
        print(f"[{stock_idx}/{self.total_stocks}] 處理：{stock_name}")
        print(f"{'='*70}")
    
    def pso_progress(self, iteration, total, best_mse):
        if iteration % 5 == 0 or iteration == total:
            percent = (iteration / total) * 100
            print(f"PSO Optimization: {percent:>5.1f}% | Best MSE: {best_mse:.6f}")
    
    def end_stock(self, stock_name):
        stock_time = time.time() - self.stock_start_time
        print(f"\n{'='*70}")
        print(f"{stock_name} 完成！耗時 {stock_time/60:.1f} 分鐘")
        print(f"{'='*70}")

tracker = None

# ============================================================
# 核心函數
# ============================================================

def process_data(X):
    que = deque(maxlen=window)
    x = []
    for i in X:
        que.append(i)
        if len(que) == window:
            x.append(list(que))
    return np.array(x)

def build_bigru_model(params, input_shape):
    units, dropout_rate, _ = params
    model = Sequential([
        Bidirectional(GRU(int(units), activation='relu', return_sequences=False), 
                       input_shape=input_shape),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mse'])
    return model

def fitness_function(params, x_train, y_train, x_val, y_val):
    try:
        model = build_bigru_model(params, x_train.shape[1:])
        model.fit(x_train, y_train, batch_size=32, epochs=8, verbose=0, shuffle=False)
        y_pred = model.predict(x_val, verbose=0)
        return mean_squared_error(y_val, y_pred)
    except:
        return float('inf')

def PSO(x_train, y_train, x_val, y_val, pop_size=8, max_iter=25):
    global tracker
    param_bounds = np.array([[32, 96], [0.1, 0.4], [0.0005, 0.005]])
    population = np.random.uniform(param_bounds[:, 0], param_bounds[:, 1], (pop_size, 3))
    velocities = np.random.uniform(-1, 1, (pop_size, 3))
    pbest_pos = population.copy()
    pbest_score = np.full(pop_size, float('inf'))
    gbest_pos = np.zeros(3)
    gbest_score = float('inf')
    
    for t in range(max_iter):
        for i in range(pop_size):
            fitness = fitness_function(population[i], x_train, y_train, x_val, y_val)
            if fitness < pbest_score[i]:
                pbest_score[i] = fitness
                pbest_pos[i] = population[i]
            if fitness < gbest_score:
                gbest_score = fitness
                gbest_pos = population[i]
        
        w, c1, c2 = 0.5, 1.5, 1.5
        for i in range(pop_size):
            r1, r2 = np.random.rand(2)
            velocities[i] = (w * velocities[i] + c1 * r1 * (pbest_pos[i] - population[i]) + c2 * r2 * (gbest_pos - population[i]))
            population[i] = np.clip(population[i] + velocities[i], param_bounds[:, 0], param_bounds[:, 1])
        if tracker:
            tracker.pso_progress(t+1, max_iter, gbest_score)
    
    print(f"\n PSO Optimal Params: units={int(gbest_pos[0])}, dropout={gbest_pos[1]:.3f}")
    return gbest_pos

def train_pso_bigru(x_train, y_train, x_test, y_test, test_indices, dates, stock_name):
    global best_result_buffer
    best_params = PSO(x_train, y_train, x_test, y_test)
    model = Sequential([
        Bidirectional(GRU(int(best_params[0]), activation='relu', return_sequences=False),
                       input_shape=x_train.shape[1:]),
        Dropout(best_params[1]),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss=loss)
    model.fit(x_train, y_train, batch_size=32, epochs=30, validation_split=0.1, verbose=0, shuffle=False)
    
    y_pred = model.predict(x_test, verbose=0)
    mse = mean_squared_error(y_test, y_pred)
    measure[0] = [mse, math.sqrt(mse), mean_absolute_error(y_test, y_pred), mean_absolute_percentage_error(y_test, y_pred), r2_score(y_test, y_pred)]
    
    y_pred_inv = scaler.inverse_transform(y_pred)
    y_test_inv = scaler.inverse_transform(y_test)
    
    base_dates = [dates[idx] for idx in test_indices]
    best_result_buffer = pd.DataFrame({
        'stock_name': stock_name, 'predict_date': base_dates,
        'predicted_close': y_pred_inv[:, 0], 'actual_close': y_test_inv[:, 0]
    })
    return model

def fit_pso_bigru(df, dates, stock_name):
    X = df.iloc[:, :-1].apply(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-7))
    y = scaler.fit_transform(df['Close'].values[:-window+1].reshape(-1, 1))
    x = process_data(X.values)
    split_point = int(len(y) * 0.7)
    model = train_pso_bigru(x[:split_point], y[:split_point], x[split_point:], y[split_point:], 
                           list(range(split_point, len(y))), dates[:-window+1].reset_index(drop=True), stock_name)
    return model

def main(data, stock_name):
    df_full = pd.read_csv(f'./data/{data}')
    dates = pd.to_datetime(df_full['Date']) if 'Date' in df_full.columns else pd.date_range('2021-01-01', periods=len(df_full), freq='D')
    fit_pso_bigru(df_full.fillna(0).sort_index().drop(columns=['Date'], errors='ignore'), dates, stock_name)

if __name__ == '__main__':
    stock_list = ['台積電', '長榮', '聯發科', '統一超']
    tracker = ProgressTracker(len(stock_list))
    for idx, stock in enumerate(stock_list, 1):
        tracker.start_stock(stock, idx)
        main(f"{stock}_date.csv", stock)
        tracker.end_stock(stock)